# 컬럼 네비게이터 — 성남시 자영업자 분석

**목적**: 병합된 데이터셋의 컬럼 구조를 파악하고, 분석에 필요한 핵심 컬럼 27개를 확정합니다.
**생성일**: 2026-04-07

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

BASE = '/sessions/hopeful-elegant-davinci/mnt/성남시/'
print('라이브러리 로드 완료')

## 1. 데이터셋 로드

In [ ]:
master = pd.read_csv(BASE + 'master_district_monthly.csv')
card_mer = pd.read_csv(BASE + 'merge_card_merchant.csv', nrows=5000)
sales = pd.read_csv(BASE + 'merge_card_sales_monthly.csv')
credit = pd.read_csv(BASE + 'merge_credit_info.csv')

datasets = {
    'master_district_monthly': master,
    'merge_card_merchant': card_mer,
    'merge_card_sales_monthly': sales,
    'merge_credit_info': credit
}

for name, df in datasets.items():
    print(f'{name}: {df.shape}')

## 2. 컬럼 품질 분석

In [ ]:
def analyze_columns(df, dataset_name):
    rows = []
    for col in df.columns:
        missing_rate = df[col].isna().mean() * 100
        unique_cnt = df[col].nunique()
        unique_ratio = unique_cnt / len(df) * 100
        quality = round(100 - missing_rate * 0.5, 1)
        sample = df[col].dropna().head(2).tolist()
        rows.append({
            'dataset': dataset_name,
            '컬럼명': col,
            'dtype': str(df[col].dtype),
            '결측률': round(missing_rate, 1),
            '유니크수': unique_cnt,
            '유니크비율': round(unique_ratio, 1),
            '품질점수': quality,
            '샘플값': str(sample)
        })
    return pd.DataFrame(rows)

result = pd.concat([analyze_columns(df, name) for name, df in datasets.items()])
print(f'전체 컬럼 수: {len(result)}')
result.head(10)

## 3. 결측률 이슈 컬럼 확인

In [ ]:
issue_cols = result[result['결측률'] > 0].sort_values('결측률', ascending=False)
print('결측률 있는 컬럼:')
display(issue_cols[['dataset', '컬럼명', '결측률', '품질점수']])

## 4. 고상관 컬럼 탐지 (마스터 데이터셋 기준)

In [ ]:
num_cols = master.select_dtypes(include='number').columns.tolist()
corr = master[num_cols].corr()

# 0.95 이상 고상관 쌍 탐지
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        val = abs(corr.iloc[i, j])
        if val >= 0.95:
            high_corr_pairs.append({
                '컬럼A': corr.columns[i],
                '컬럼B': corr.columns[j],
                '상관계수': round(val, 3)
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('상관계수', ascending=False)
print(f'고상관 쌍 수 (r≥0.95): {len(high_corr_df)}')
display(high_corr_df)

## 5. 상관관계 히트맵

In [ ]:
CORE_COLUMNS = [
    'ym', 'mer_total', 'mer_open', 'mer_close', 'mer_net', 'close_rate', 'mer_op_5yp',
    'sales_amt', 'sales_cnt', 'sales_per_merchant',
    'in_self_ent', 'out_self_ent', 'self_ent_net', 'in_avg_inc', 'out_avg_inc',
    'sum_income', 'sum_loan', 'sum_hous_loan', 'new_hous_cnt', 'new_hous_rate',
    'over_loan_cnt', 'loan_delinq_rate', 'score_high', 'score_low', 'econ_cnt', 'total_inflow'
]

valid_cols = [c for c in CORE_COLUMNS if c in master.columns]
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 가맹점·매출·유동인구
group1 = [c for c in valid_cols if c in ['mer_total','mer_open','mer_close','mer_net','close_rate',
                                           'mer_op_5yp','sales_amt','sales_cnt','sales_per_merchant','total_inflow']]
corr1 = master[group1].corr()
sns.heatmap(corr1, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('가맹점·매출·유동인구 상관관계', fontsize=13, fontweight='bold')

# 신용·자영업자 이동
group2 = [c for c in valid_cols if c in ['in_self_ent','out_self_ent','self_ent_net','in_avg_inc',
                                          'sum_income','sum_loan','sum_hous_loan','over_loan_cnt',
                                          'loan_delinq_rate','score_high','score_low']]
corr2 = master[group2].dropna().corr()
sns.heatmap(corr2, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=axes[1], cbar_kws={'shrink': 0.8})
axes[1].set_title('신용·자영업자 이동 상관관계', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(BASE + 'column_corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('히트맵 저장 완료')

## 6. 최종 핵심 컬럼 27개 확정

In [ ]:
FINAL_CORE = [
    # 키
    'ym', 'gu_nm',
    # 가맹점 생존 지표
    'mer_total', 'mer_open', 'mer_close', 'mer_net', 'close_rate', 'mer_op_5yp',
    # 매출 지표
    'sales_amt', 'sales_cnt', 'sales_per_merchant',
    # 자영업자 이동
    'in_self_ent', 'out_self_ent', 'self_ent_net', 'in_avg_inc', 'out_avg_inc',
    # 지역경제 / 젠트리피케이션 대리지표
    'sum_income', 'sum_loan', 'sum_hous_loan', 'new_hous_cnt', 'new_hous_rate',
    'over_loan_cnt', 'loan_delinq_rate',
    # 신용 구조
    'score_high', 'score_low', 'econ_cnt',
    # 유동인구
    'total_inflow'
]

print(f'✅ 최종 EDA 핵심 컬럼: {len(FINAL_CORE)}개')
print(f'❌ 제거 컬럼: {40 - len(FINAL_CORE)}개')
print()

# 구별 요약 통계
summary = master.groupby('gu_nm')[[
    'mer_total', 'mer_open', 'mer_close', 'mer_net',
    'sales_amt', 'in_self_ent', 'out_self_ent', 'self_ent_net', 'loan_delinq_rate'
]].mean().round(1)

print('=== 구별 월평균 핵심 지표 ===')
display(summary)

## 요약

- **전체 컬럼 109개** 중 **27개 핵심 컬럼** 확정
- **제거 권고 7개**: mer_stop, mer_fran, mer_op_lt1y~4_5y (고상관 중복)
- **주의 컬럼**: out_tot, out_self_ent, out_avg_inc (결측률 33.3%)
- **다음 단계**: `programmatic-eda`로 27개 컬럼 탐색적 분석 수행